# 05 Train Improved WLASL100 Model

This notebook improves the WLASL100 baseline model. It uses global train-set normalisation, velocity features, balanced sampling, a BiGRU with temporal attention, AdamW, learning-rate scheduling, gradient clipping, early stopping, and Top-1/Top-3/Top-5/Macro-F1 metrics.

The goal is to build a stronger WLASL100 model before scaling to WLASL300, WLASL1000, and WLASL2000.

In [ ]:
from pathlib import Path
import json
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path("E:/Be_My_Ear")

BASE_DIR = PROJECT_ROOT / "data" / "processed" / "ASL" / "WLASL100"
CLEAN_INDEX_FILE = BASE_DIR / "wlasl100_clean_keypoint_index.csv"
LABEL_MAP_FILE = PROJECT_ROOT / "data" / "label_maps" / "asl_wlasl100_labels.json"

MODEL_DIR = PROJECT_ROOT / "models" / "ASL"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "improved_bigru_attention_wlasl100.pt"
HISTORY_PATH = MODEL_DIR / "improved_bigru_attention_wlasl100_history.csv"
NORM_STATS_PATH = MODEL_DIR / "wlasl100_train_norm_stats.npz"

print("Clean index exists:", CLEAN_INDEX_FILE.exists())
print("Label map exists:", LABEL_MAP_FILE.exists())
print("Model will save to:", MODEL_PATH)

In [ ]:
df = pd.read_csv(CLEAN_INDEX_FILE)

print("Total clean samples:", len(df))
print("Total classes:", df["label_id"].nunique())
print("Input shape per file should be: (60, 258)")
print("Example keypoint path:", df.iloc[0]["keypoint_path"])

df.head()

## 1. Reproducible class-balanced split

Every class is kept in train, validation, and test where possible. Because WLASL100 has few samples per class, validation/test scores can be noisy, but this split gives us consistent comparison between model versions.

In [ ]:
train_records = []
val_records = []
test_records = []

for label_id, group in df.groupby("label_id"):
    group = group.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n = len(group)

    n_test = max(1, int(round(n * 0.15)))
    n_val = max(1, int(round(n * 0.15)))

    test_part = group.iloc[:n_test]
    val_part = group.iloc[n_test:n_test + n_val]
    train_part = group.iloc[n_test + n_val:]

    train_records.append(train_part)
    val_records.append(val_part)
    test_records.append(test_part)

train_df = pd.concat(train_records).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = pd.concat(val_records).sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = pd.concat(test_records).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))
print("Train classes:", train_df["label_id"].nunique())
print("Validation classes:", val_df["label_id"].nunique())
print("Test classes:", test_df["label_id"].nunique())

In [ ]:
train_counts = train_df["label_id"].value_counts().sort_index()

print("Train class distribution summary:")
print("Min:", train_counts.min())
print("Max:", train_counts.max())
print("Mean:", round(train_counts.mean(), 2))

train_counts.head()

## 2. Compute global normalisation statistics from training data only

This avoids leaking validation/test information into training. The same train-set mean and standard deviation are used for train, validation, test, and future prediction.

In [ ]:
def compute_train_normalisation_stats(train_dataframe):
    total_sum = None
    total_sq_sum = None
    total_count = 0

    for path in tqdm(train_dataframe["keypoint_path"], desc="Computing train mean/std"):
        arr = np.load(path).astype(np.float32)

        if total_sum is None:
            total_sum = arr.sum(axis=0)
            total_sq_sum = (arr ** 2).sum(axis=0)
        else:
            total_sum += arr.sum(axis=0)
            total_sq_sum += (arr ** 2).sum(axis=0)

        total_count += arr.shape[0]

    mean = total_sum / total_count
    variance = (total_sq_sum / total_count) - (mean ** 2)
    variance = np.maximum(variance, 1e-6)
    std = np.sqrt(variance)

    return mean.astype(np.float32), std.astype(np.float32)

train_mean, train_std = compute_train_normalisation_stats(train_df)

np.savez(NORM_STATS_PATH, mean=train_mean, std=train_std)

print("Saved normalisation stats to:", NORM_STATS_PATH)
print("Mean shape:", train_mean.shape)
print("Std shape:", train_std.shape)

## 3. Dataset with velocity features

Each keypoint file starts as `(60, 258)`. After normalisation, we add frame-to-frame velocity features. The final input becomes `(60, 516)`:

`258 original features + 258 velocity features = 516 features`.

In [ ]:
class ImprovedSignKeypointDataset(Dataset):
    def __init__(self, dataframe, mean, std, use_velocity=True):
        self.dataframe = dataframe.reset_index(drop=True)
        self.mean = mean.reshape(1, -1).astype(np.float32)
        self.std = std.reshape(1, -1).astype(np.float32)
        self.use_velocity = use_velocity

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        keypoints = np.load(row["keypoint_path"]).astype(np.float32)

        keypoints = (keypoints - self.mean) / (self.std + 1e-6)

        if self.use_velocity:
            velocity = np.zeros_like(keypoints, dtype=np.float32)
            velocity[1:] = keypoints[1:] - keypoints[:-1]
            features = np.concatenate([keypoints, velocity], axis=1)
        else:
            features = keypoints

        label = int(row["label_id"])

        return torch.tensor(features, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

In [ ]:
BATCH_SIZE = 32
USE_VELOCITY = True
INPUT_SIZE = 516 if USE_VELOCITY else 258

train_dataset = ImprovedSignKeypointDataset(train_df, train_mean, train_std, use_velocity=USE_VELOCITY)
val_dataset = ImprovedSignKeypointDataset(val_df, train_mean, train_std, use_velocity=USE_VELOCITY)
test_dataset = ImprovedSignKeypointDataset(test_df, train_mean, train_std, use_velocity=USE_VELOCITY)

x_sample, y_sample = train_dataset[0]
print("Single sample feature shape:", x_sample.shape)
print("Single sample label:", y_sample.item())
print("Input size:", INPUT_SIZE)

## 4. Balanced sampling

Some signs have more examples than others. `WeightedRandomSampler` helps the model see underrepresented classes more often during training.

In [ ]:
train_label_counts = train_df["label_id"].value_counts().sort_index()
class_sample_counts = train_label_counts.to_dict()

sample_weights = train_df["label_id"].map(lambda label: 1.0 / class_sample_counts[label]).values
sample_weights = torch.DoubleTensor(sample_weights)

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

x_batch, y_batch = next(iter(train_loader))

print("Input batch shape:", x_batch.shape)
print("Label batch shape:", y_batch.shape)
print("Batch labels:", y_batch[:10])

## 5. Improved model: BiGRU + Temporal Attention

The baseline used the last LSTM output. This model uses attention pooling so it can learn which frames matter most for each sign.

In [ ]:
class BiGRUAttentionModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=2, dropout=0.4):
        super(BiGRUAttentionModel, self).__init__()

        self.input_projection = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.gru = nn.GRU(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        x = self.input_projection(x)
        gru_out, _ = self.gru(x)

        attention_scores = self.attention(gru_out).squeeze(-1)
        attention_weights = torch.softmax(attention_scores, dim=1).unsqueeze(-1)

        context = torch.sum(gru_out * attention_weights, dim=1)
        logits = self.classifier(context)

        return logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

NUM_CLASSES = df["label_id"].nunique()

model = BiGRUAttentionModel(
    input_size=INPUT_SIZE,
    hidden_size=256,
    num_classes=NUM_CLASSES,
    num_layers=2,
    dropout=0.4
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=5
)

print(model)

## 6. Metrics and training helpers

In [ ]:
def top_k_accuracy(outputs, labels, k=5):
    _, top_k_preds = outputs.topk(k, dim=1)
    correct = top_k_preds.eq(labels.view(-1, 1).expand_as(top_k_preds))
    return correct.any(dim=1).float().mean().item()

def get_current_lr(optimizer):
    return optimizer.param_groups[0]["lr"]

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, epoch, total_epochs):
    model.train()

    total_loss = 0
    total_top1 = 0
    total_top3 = 0
    total_top5 = 0

    all_preds = []
    all_labels = []

    progress_bar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} [Training]", leave=False)

    for step, (x, y) in enumerate(progress_bar, start=1):
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        preds = torch.argmax(outputs, dim=1)

        batch_top1 = (preds == y).float().mean().item()
        batch_top3 = top_k_accuracy(outputs, y, k=3)
        batch_top5 = top_k_accuracy(outputs, y, k=5)

        total_loss += loss.item()
        total_top1 += batch_top1
        total_top3 += batch_top3
        total_top5 += batch_top5

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

        progress_bar.set_postfix({
            "step": f"{step}/{len(loader)}",
            "loss": f"{loss.item():.4f}",
            "top1": f"{batch_top1:.4f}",
            "top5": f"{batch_top5:.4f}",
            "lr": f"{get_current_lr(optimizer):.6f}"
        })

    avg_loss = total_loss / len(loader)
    avg_top1 = total_top1 / len(loader)
    avg_top3 = total_top3 / len(loader)
    avg_top5 = total_top5 / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    return avg_loss, avg_top1, avg_top3, avg_top5, macro_f1


def evaluate(model, loader, criterion, device, epoch, total_epochs, phase="Validation"):
    model.eval()

    total_loss = 0
    total_top1 = 0
    total_top3 = 0
    total_top5 = 0

    all_preds = []
    all_labels = []

    progress_bar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} [{phase}]", leave=False)

    with torch.no_grad():
        for step, (x, y) in enumerate(progress_bar, start=1):
            x = x.to(device)
            y = y.to(device)

            outputs = model(x)
            loss = criterion(outputs, y)

            preds = torch.argmax(outputs, dim=1)

            batch_top1 = (preds == y).float().mean().item()
            batch_top3 = top_k_accuracy(outputs, y, k=3)
            batch_top5 = top_k_accuracy(outputs, y, k=5)

            total_loss += loss.item()
            total_top1 += batch_top1
            total_top3 += batch_top3
            total_top5 += batch_top5

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

            progress_bar.set_postfix({
                "step": f"{step}/{len(loader)}",
                "loss": f"{loss.item():.4f}",
                "top1": f"{batch_top1:.4f}",
                "top5": f"{batch_top5:.4f}"
            })

    avg_loss = total_loss / len(loader)
    avg_top1 = total_top1 / len(loader)
    avg_top3 = total_top3 / len(loader)
    avg_top5 = total_top5 / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    return avg_loss, avg_top1, avg_top3, avg_top5, macro_f1

## 7. Train improved model

Start with 60 epochs and early stopping. If your laptop is slow, reduce `EPOCHS` to 30 first.

In [ ]:
EPOCHS = 60
EARLY_STOPPING_PATIENCE = 15

history = {
    "train_loss": [],
    "train_top1": [],
    "train_top3": [],
    "train_top5": [],
    "train_f1": [],
    "val_loss": [],
    "val_top1": [],
    "val_top3": [],
    "val_top5": [],
    "val_f1": [],
    "lr": []
}

best_val_f1 = 0.0
best_val_top5 = 0.0
epochs_without_improvement = 0

print("=" * 80)
print("Be My Ear - Improved WLASL100 Training")
print("=" * 80)
print(f"Device: {device}")
print(f"Model: BiGRU + Temporal Attention")
print(f"Input shape: (60, {INPUT_SIZE})")
print(f"Classes: {NUM_CLASSES}")
print(f"Train samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Balanced sampler: Yes")
print(f"Velocity features: {USE_VELOCITY}")
print(f"Model save path: {MODEL_PATH}")
print("=" * 80)

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("-" * 80)

    train_loss, train_top1, train_top3, train_top5, train_f1 = train_one_epoch(
        model, train_loader, optimizer, criterion, device, epoch, EPOCHS
    )

    val_loss, val_top1, val_top3, val_top5, val_f1 = evaluate(
        model, val_loader, criterion, device, epoch, EPOCHS, phase="Validation"
    )

    scheduler.step(val_f1)
    current_lr = get_current_lr(optimizer)

    history["train_loss"].append(train_loss)
    history["train_top1"].append(train_top1)
    history["train_top3"].append(train_top3)
    history["train_top5"].append(train_top5)
    history["train_f1"].append(train_f1)

    history["val_loss"].append(val_loss)
    history["val_top1"].append(val_top1)
    history["val_top3"].append(val_top3)
    history["val_top5"].append(val_top5)
    history["val_f1"].append(val_f1)
    history["lr"].append(current_lr)

    improved = val_f1 > best_val_f1

    if improved:
        best_val_f1 = val_f1
        best_val_top5 = val_top5
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_f1": best_val_f1,
            "best_val_top5": best_val_top5,
            "num_classes": NUM_CLASSES,
            "input_size": INPUT_SIZE,
            "sequence_length": 60,
            "use_velocity": USE_VELOCITY,
            "architecture": "BiGRUAttentionModel"
        }, MODEL_PATH)

        save_status = "Saved new best model"
    else:
        epochs_without_improvement += 1
        save_status = "No improvement"

    print(
        f"Train | Loss: {train_loss:.4f} | Top-1: {train_top1:.4f} | "
        f"Top-3: {train_top3:.4f} | Top-5: {train_top5:.4f} | F1: {train_f1:.4f}"
    )

    print(
        f"Val   | Loss: {val_loss:.4f} | Top-1: {val_top1:.4f} | "
        f"Top-3: {val_top3:.4f} | Top-5: {val_top5:.4f} | F1: {val_f1:.4f}"
    )

    print(f"Learning rate: {current_lr:.8f}")
    print(f"Status: {save_status}")
    print(f"Best Val F1 so far: {best_val_f1:.4f}")
    print(f"Best Val Top-5 so far: {best_val_top5:.4f}")
    print(f"Epochs without improvement: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("\nEarly stopping triggered.")
        break

end_time = time.time()
training_minutes = (end_time - start_time) / 60

print("\n" + "=" * 80)
print("Training completed")
print("=" * 80)
print(f"Total training time: {training_minutes:.2f} minutes")
print(f"Best validation F1: {best_val_f1:.4f}")
print(f"Best validation Top-5: {best_val_top5:.4f}")
print(f"Best model saved to: {MODEL_PATH}")
print("=" * 80)

## 8. Save history and plot training curves

In [ ]:
history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)

print("Saved training history to:")
print(HISTORY_PATH)

history_df.head()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history_df["train_loss"], label="Train Loss")
plt.plot(history_df["val_loss"], label="Validation Loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history_df["train_top1"], label="Train Top-1")
plt.plot(history_df["val_top1"], label="Validation Top-1")
plt.plot(history_df["train_top5"], label="Train Top-5")
plt.plot(history_df["val_top5"], label="Validation Top-5")
plt.title("Top-1 and Top-5 Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history_df["train_f1"], label="Train Macro F1")
plt.plot(history_df["val_f1"], label="Validation Macro F1")
plt.title("Macro F1 Score")
plt.xlabel("Epoch")
plt.ylabel("F1")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history_df["lr"], label="Learning Rate")
plt.title("Learning Rate Schedule")
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Test set evaluation

In [ ]:
checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

print("Loaded best model from epoch:", checkpoint["epoch"])
print("Best validation F1:", checkpoint["best_val_f1"])
print("Best validation Top-5:", checkpoint["best_val_top5"])

In [ ]:
def collect_predictions(model, loader, device):
    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Collecting predictions"):
            x = x.to(device)
            y = y.to(device)

            outputs = model(x)
            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_labels.extend(y.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

In [ ]:
y_true, y_pred, y_probs = collect_predictions(model, test_loader, device)

test_top1 = accuracy_score(y_true, y_pred)
test_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

def numpy_top_k_accuracy(y_true, y_probs, k):
    correct = 0
    for true_label, prob in zip(y_true, y_probs):
        top_k_preds = np.argsort(prob)[-k:]
        if true_label in top_k_preds:
            correct += 1
    return correct / len(y_true)

test_top3 = numpy_top_k_accuracy(y_true, y_probs, k=3)
test_top5 = numpy_top_k_accuracy(y_true, y_probs, k=5)

print("=" * 80)
print("Test Set Evaluation")
print("=" * 80)
print(f"Test Top-1 Accuracy: {test_top1:.4f}")
print(f"Test Top-3 Accuracy: {test_top3:.4f}")
print(f"Test Top-5 Accuracy: {test_top5:.4f}")
print(f"Test Macro F1: {test_f1:.4f}")
print("=" * 80)

## 10. Prediction examples

In [ ]:
with open(LABEL_MAP_FILE, "r", encoding="utf-8") as f:
    label_map = json.load(f)

test_examples = test_df.reset_index(drop=True).copy()
example_rows = []

for i in range(min(15, len(test_examples))):
    true_id = int(y_true[i])
    pred_id = int(y_pred[i])
    confidence = float(y_probs[i][pred_id])

    top5_ids = np.argsort(y_probs[i])[-5:][::-1]
    top5_glosses = [
        label_map[str(label_id)]["gloss"] if str(label_id) in label_map else str(label_id)
        for label_id in top5_ids
    ]

    example_rows.append({
        "video_id": test_examples.iloc[i]["video_id"],
        "true_gloss": label_map[str(true_id)]["gloss"] if str(true_id) in label_map else true_id,
        "predicted_gloss": label_map[str(pred_id)]["gloss"] if str(pred_id) in label_map else pred_id,
        "confidence": round(confidence, 4),
        "top5_predictions": ", ".join(top5_glosses)
    })

pd.DataFrame(example_rows)

## 11. Final summary

In [ ]:
print("Final improved WLASL100 model summary")
print("------------------------------------")
print(f"Dataset: WLASL100")
print(f"Clean samples: {len(df)}")
print(f"Classes: {NUM_CLASSES}")
print(f"Model: BiGRU + Temporal Attention")
print(f"Input shape: (60, {INPUT_SIZE})")
print(f"Velocity features: {USE_VELOCITY}")
print(f"Best validation F1: {best_val_f1:.4f}")
print(f"Best validation Top-5: {best_val_top5:.4f}")
print(f"Test Top-1 Accuracy: {test_top1:.4f}")
print(f"Test Top-3 Accuracy: {test_top3:.4f}")
print(f"Test Top-5 Accuracy: {test_top5:.4f}")
print(f"Test Macro F1: {test_f1:.4f}")
print(f"Model saved to: {MODEL_PATH}")
print(f"History saved to: {HISTORY_PATH}")
print(f"Normalisation stats saved to: {NORM_STATS_PATH}")